# GPAP — GPU SAT Sampling Benchmark (cross-GPU)

Measure **how fast GPAP draws guaranteed-valid SAT models on *this* GPU vs CMSGen**, the SOTA
fast "uniform-like" CPU sampler.

GPAP runs CMSGen's own algorithm — *random variable → random polarity → unit-propagate* — but
**K trajectories in parallel on the GPU**, discarding conflicted ones (every full, conflict-free
trajectory is a model: all clause `sat_count > 0`). Model-preserving unit-propagation
simplification shrinks the formula so a much larger `K` fits — the throughput lever.

Three measurements on the projected-sampling `sk_` benchmarks (independent support via `c ind`):

1. **Throughput vs CMSGen** — GPAP samples/s ÷ CMSGen samples/s (end-to-end wall), per niche
   instance + geomean.
2. **Quality parity** — 2-wise coverage % (the combinatorial-testing metric); the speed win must
   come at equal coverage.
3. **K-scaling** — sweep `K` on a couple of instances: throughput is VRAM-limited so it scales
   ~linearly with `K`. **This is the A100 story** — an 80 GB GPU sits far up the curve.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4/L4/A100…). `K` auto-sizes to the
GPU's VRAM — no manual tuning. Results save per-GPU to `benchmarks/colab_sampling_results/`.

**Idle-safe layout:** the benchmark runs **one circuit family per cell** (ISCAS-89, ITC'99,
ISCAS-85), each saving every instance to a partial file as it finishes. If Colab disconnects a
long-idle session, you lose at most the in-flight instance — re-run the cell to **resume**
(finished instances are skipped). A final `finalize_families()` cell merges + plots.


## 1. Setup — clone repo + build CMSGen
Torch ships preinstalled on Colab. We add `python-sat`, then build **CMSGen** from source
(meelgroup/cmsgen; no pip wheel) so we have the head-to-head competitor.

In [ ]:
import os, subprocess, sys
from getpass import getpass

REPO_OWNER = "yogevshalmon"
REPO_NAME  = "gpap"
BRANCH     = "custom_kernel"   # the branch with the fused kernel + mining + decision stats

# Preferred: store your PAT as a Colab Secret named GITHUB_TOKEN (left sidebar -> key icon).
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    token = ""
if not token:
    print("No GITHUB_TOKEN secret found.")
    print("Create one at GitHub -> Settings -> Developer settings -> Personal access tokens")
    token = getpass("GitHub PAT (leave blank if repo is public): ")

auth     = f"{token}@" if token else ""
repo_url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"

if os.path.basename(os.getcwd()) != REPO_NAME:
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, REPO_NAME],
                       check=True)
    os.chdir(REPO_NAME)
subprocess.run(["git", "pull", "--ff-only", repo_url, BRANCH], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "python-sat", "tqdm"], check=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("repo:", os.getcwd(), "| branch:", BRANCH)

In [ ]:
# Build CMSGen (the comparison sampler). Needs cmake + build-essential + zlib.
# Skip this cell to run GPAP-only (the K-scaling + throughput numbers still work).
subprocess.run(["apt-get", "install", "-y", "-q", "cmake", "build-essential", "zlib1g-dev"],
               check=False)
from scripts.colab_sampling_benchmark import build_cmsgen
CMSGEN = build_cmsgen()
print("cmsgen:", CMSGEN or "(build failed — running GPAP-only)")

## 2. Detect the GPU
Confirms which accelerator this run used (recorded in the result file). The A100's large VRAM is
what drives a large auto-`K`.

In [ ]:
import torch
from scripts.colab_sampling_benchmark import (
    run_family, finalize_families, run_ksweep, compare_saved, FAMILIES,
)
from scripts.colab_benchmark import gpu_info

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
info = gpu_info(device)
for k, v in info.items():
    print(f"{k:>16}: {v}")
if info.get("total_gb") is None:
    print("\n  No CUDA GPU detected — set Runtime > Change runtime type > GPU and re-run cell 1.")

# ---- shared run config (all family cells read these) ----
QUICK  = False        # False = full suite (the A100 run). True = fast smoke subset (minutes).
CYCLES = [1, 5, 10, 25]   # unroll depths for the sequential families (ISCAS-89 / ITC'99)
print("\nfamilies:", list(FAMILIES), "| QUICK =", QUICK)


## 3. Run the benchmark — one family per cell (idle-safe)

Each cell below benchmarks **one circuit family** vs CMSGen and **saves every instance as it
finishes** (to `benchmarks/colab_sampling_results/_partial/`). Run them top-to-bottom (or
`Runtime ▸ Run all`). If a cell is interrupted, just **re-run it** — already-finished instances
are skipped and it picks up where it left off.

- **`iscas89`** — s386/s526/s832/s1196, unrolled over `CYCLES` (the Recurrent-CircuitSAT family).
- **`itc99`** — b01–b14, unrolled over `CYCLES`.
- **`iscas85`** — c432–c7552 combinational (T=1, one pinned output, the DEMOTIC family).

`K` auto-sizes to all VRAM, so an A100 lands far up the K-scaling curve. Each row reports
throughput vs CMSGen, a UNIQUE-solutions count, and **decisions/s + implications/s** (the
CDCL-comparable work rates: implications = assignments − decisions).

> **Keeping the session alive:** Colab reaps runtimes it thinks are idle. Keep the browser tab
> foregrounded (a backgrounded tab gets throttled and disconnected). If you must walk away, the
> per-instance saves mean a re-run resumes — nothing is lost as long as the runtime's disk
> survives. On Colab Pro the idle window is longer.

In [ ]:
# ISCAS-89 sequential (s*), unrolled over CYCLES. ~minutes on quick; longer on the full sweep.
p89 = run_family("iscas89", quick=QUICK, cmsgen_bin=CMSGEN, circuit_cycles=CYCLES)

In [ ]:
# ITC'99 (b01-b14), unrolled over CYCLES. The largest family — re-run to resume if interrupted.
p99 = run_family("itc99", quick=QUICK, cmsgen_bin=CMSGEN, circuit_cycles=CYCLES)

In [ ]:
# ISCAS-85 combinational (c*), T=1, one pinned output (DEMOTIC setup).
p85 = run_family("iscas85", quick=QUICK, cmsgen_bin=CMSGEN)

In [ ]:
# OPTIONAL — the sk_ projected-sampling suite (not a circuit family). Uncomment to include it.
# psk = run_family("sk", quick=QUICK, cmsgen_bin=CMSGEN)

## 4. Merge families → plots + saved result

Run **after** the family cells. `finalize_families()` merges all the partial files for this GPU
into the single `benchmarks/colab_sampling_results/<gpu>.json` that the cross-GPU comparison and
the plots read, prints the headline (peak throughput + decisions/s + implications/s), and renders
the plots inline.

In [ ]:
payload = finalize_families(device=device, do_plots=True)

# Show the saved plots inline + offer a download of the combined result.
from scripts.colab_sampling_benchmark import DEFAULT_OUT_DIR, save_results
from scripts.colab_sampling_plots import make_plots
from IPython.display import Image, display
for _p in make_plots(payload, DEFAULT_OUT_DIR):
    display(Image(filename=_p))
try:
    from google.colab import files
    files.download(save_results(payload))
except Exception:
    pass

## 4b. K-scaling sweep — the A100 headroom story

Sweep `K` on the highest-throughput circuit: throughput is VRAM-limited, so it scales ~linearly
with `K`. An A100 (40/80 GB) sits far up this curve vs a T4/L4 — the point of running on a bigger
GPU. Merges the sweep into the saved `<gpu>.json` and re-plots.

In [ ]:
# Sweeps the best circuit from finalize (pass circuit="b03", cycles=25 to pick a specific one).
payload = run_ksweep(payload, device=device, do_plots=True)

## 5. Compare GPUs

Rank every saved per-GPU result by peak throughput (reads the combined `<gpu>.json` files, not the partials).

In [ ]:
_ = compare_saved()

## 6. (Optional) Commit the result back to the repo
Persist this GPU's numbers in version control so they accumulate across GPUs. Needs a GitHub
**Personal Access Token** with `repo` write scope. Uncomment, fill in, and run:

In [ ]:
# from getpass import getpass
# if not token:  # reuse token from cell 1 if already set
#     token = getpass("GitHub PAT: ")
# GIT_NAME, GIT_EMAIL = "Your Name", "you@example.com"
#
# subprocess.run(["git", "config", "user.name", GIT_NAME], check=True)
# subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)
# # subprocess.run(["git", "add", "benchmarks/colab_sampling_results/*.json"], check=True)  # combined only, not _partial/
# subprocess.run(["git", "commit", "-m", f"colab sampling: {payload['gpu']['name']} result"], check=True)
# remote = f"https://{token}@github.com/yogevshalmon/gpap.git"
# subprocess.run(["git", "push", remote, f"HEAD:{BRANCH}"], check=True)
# print("pushed")